# 08 — Context Managers and Resource Safety

Context managers provide structured setup and cleanup. They are essential for file handles, locks, temporary resources, and database transactions. Understand both with blocks and contextlib.contextmanager.

*Intermediate · 75–120 min*

## You will learn

- Explain the __enter__/__exit__ protocol
- Use with for safe resource cleanup
- Build a context manager with contextlib.contextmanager
- Handle failure without hiding exceptions

In [ ]:
# Setup: run this first. On Google Colab (or anywhere the course files are missing) it
# downloads the course and moves into this notebook's folder. Elsewhere it does nothing.
import os, sys
if not os.path.exists("../README.md"):
    course = os.path.join(os.path.expanduser("~"), "python-notebook")
    if not os.path.isdir(course):
        os.system(f"git clone -q --depth 1 https://github.com/SwarmLadder/python-notebook {course}")
    os.chdir(os.path.join(course, "intermediate", "notebooks"))
    sys.path.insert(0, os.getcwd())

## Examples

### 1. The with statement

A context manager is entered before the indented block and exited afterward, even if an exception occurs. File objects close automatically using with. This avoids leaking file descriptors and makes cleanup visible.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

with TemporaryDirectory() as directory:
    path = Path(directory) / "notes.txt"
    with path.open("w", encoding="utf-8") as stream:
        stream.write("hello\nworld\n")
    print(path.read_text(encoding="utf-8"))

**Reasoning prompt:** What happens if writing raises an exception inside the with block?

### 2. A class-based context manager

__enter__ returns the object used after as; __exit__ receives exception information and runs on normal and abnormal exits. Returning False (or None) allows exceptions to propagate. Returning True suppresses them, which should be deliberate.

In [ ]:
class LabelledBlock:
    def __init__(self, label):
        self.label = label
    def __enter__(self):
        print("begin", self.label)
        return self
    def __exit__(self, exc_type, exc, traceback):
        print("end", self.label)
        return False

with LabelledBlock("report"):
    print("inside")

**Reasoning prompt:** Why is suppressing all exceptions from __exit__ generally unsafe?

### 3. The contextmanager decorator

contextlib.contextmanager turns a generator with one yield into a context manager. Code before yield sets up a resource; code in finally cleans it up. The try/finally is important if body code raises.

In [ ]:
from contextlib import contextmanager

@contextmanager
def status(label):
    print("opening:", label)
    try:
        yield {"name": label}
    finally:
        print("closing:", label)

with status("demo") as info:
    print(info["name"])

**Reasoning prompt:** Would the closing message still print if the body raises?

### 4. Temporary resources in tests

TemporaryDirectory gives you a disposable directory that is deleted upon leaving its context. Tests can safely create files without overwriting course data or leaving stray artifacts in the repository.

In [ ]:
from tempfile import TemporaryDirectory
from pathlib import Path
with TemporaryDirectory() as folder:
    temporary_file = Path(folder) / "result.csv"
    temporary_file.write_text("name,score\nAda,90\n", encoding="utf-8")
    print(temporary_file.exists(), temporary_file.read_text(encoding="utf-8").splitlines())

**Reasoning prompt:** Why is a temporary directory more reliable than using a hardcoded path in a notebook?

## Try it

Short fill-in exercises on this lesson: replace each `___`, run the cell, and `check` tells you whether you're right. The answer itself stays hidden. More practice for this topic is in the [workbook](../exercises/02_iteration_and_abstractions_exercises.ipynb).

In [ ]:
from lesson import Exercises, ___, run

check = Exercises('intermediate/08', {1: '9ef0a6527b9e0c62', 2: '93636940a89fb61b', 3: '1ce827efc5b09794', 4: '2dd445a9efc9e8e9'})

**Exercise 1.** `with open(...)` closes the file even if an error happens.

In [ ]:
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as folder:
    path = Path(folder) / "log.txt"
    with open(path, "w", encoding="utf-8") as handle:
        handle.write("saved")
    closed = ___
check(1, closed)

**Exercise 2.** A class-based context manager that records enter and exit.

In [ ]:
class Recorder:
    def __init__(self):
        self.events = []

    def __enter__(self):
        self.events.append("enter")
        return ___           # the object bound by `as`

    def __exit__(self, exc_type, exc, tb):
        self.events.append(___)
        return False          # don't hide errors

with Recorder() as r:
    r.events.append("work")
check(2, r.events)

**Exercise 3.** A generator-based context manager with `contextlib.contextmanager`.

In [ ]:
from contextlib import contextmanager

log = []

@contextmanager
def tag(name):
    log.append(f"<{___}>")
    yield                     # the with-block runs here
    log.append(f"</{___}>")

with tag("b"):
    log.append("bold")
check(3, log)

**Exercise 4.** Cleanup must run even when the block fails. Use `try/finally`.

In [ ]:
from contextlib import contextmanager

log = []

@contextmanager
def resource():
    log.append("open")
    try:
        yield
    finally:
        log.append(___)

try:
    with resource():
        raise RuntimeError("boom")
except RuntimeError:
    pass
check(4, log)

## Common mistakes

- A context manager must clean up on both success and failure.
- Returning True from __exit__ suppresses exceptions.
- Avoid hardcoded output paths that could overwrite real files.

## Quick quiz

Replace each `'?'` in the cell below with your choice and run it. The answers are hidden, but each one is checked.

**1.** When does __exit__ run?

   - **a)** Never automatically
   - **b)** Only on success
   - **c)** When the with block ends, even on an exception

**2.** If __exit__ returns True, an exception in the block is…

   - **a)** Suppressed
   - **b)** Re-raised
   - **c)** Logged

**3.** @contextlib.contextmanager turns a ___ into a context manager.

   - **a)** class
   - **b)** dict
   - **c)** generator function with one yield

In [ ]:
from quiz import Quiz

quiz = Quiz('intermediate/08', {1: 'a9c9a1f44bd6', 2: 'b0d0a46b3437', 3: '80a2b5fb6dc0'})
quiz.answer(1, '?')
quiz.answer(2, '?')
quiz.answer(3, '?')

**Next:** [09 — Functional Tools, Sorting, and itertools](09_functional_tools_and_itertools.ipynb)